# Quick Test — 단일 split SGLD + Eval + Viz

하나의 split 에 대해 빠르게:
1. VP-SGLD 샘플 생성
2. Classifier 평가 (bar chart)
3. Heatmap 시각화

전체 pipeline 을 빠르게 확인하고 싶을 때 사용.

## 0. Setup

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os, sys, json, time
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as _mp
try: _mp.set_start_method('spawn', force=True)
except RuntimeError: pass

os.chdir('/home/work/JooKyung/TabEBM')
sys.path.insert(0, 'experiments'); sys.path.insert(0, 'src')

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
import warnings; warnings.filterwarnings('ignore')

CLASSIFIER_BUILDERS = {
    'knn':     lambda s: KNeighborsClassifier(n_jobs=-1),
    'lr':      lambda s: LogisticRegression(max_iter=1000, n_jobs=-1, random_state=s),
    'rf':      lambda s: RandomForestClassifier(n_jobs=-1, random_state=s),
    'xgboost': lambda s: xgb.XGBClassifier(n_jobs=-1, eval_metric='logloss',
                                            use_label_encoder=False, random_state=s, verbosity=0),
    'mlp':     lambda s: MLPClassifier(max_iter=300, random_state=s),
}
print('ready')

ready


## 1. eval_dir + split 선택

In [2]:
fair_root = Path('experiments/fair_eval')
if fair_root.exists():
    for p in sorted(fair_root.iterdir()):
        if p.is_dir() and (p / 'config.json').exists():
            cfg = json.loads((p / 'config.json').read_text())
            print(f'  {p.name}  K={cfg["K"]} splits={cfg["n_splits"]}')

  20260417_215937  K=10 splits=10


In [3]:
EVAL_DIR = Path('experiments/fair_eval/20260417_215937')   # <-- 변경!
SPLIT = 0                                                  # <-- 테스트할 split

config = json.loads((EVAL_DIR / 'config.json').read_text())
data = np.load(EVAL_DIR / 'data.npz')
X_all, y_all = data['X_all'], data['y_all']
sp = np.load(EVAL_DIR / 'splits.npz')
tr, te = sp[f'tr_{SPLIT}'], sp[f'te_{SPLIT}']
X_tr, y_tr = X_all[tr], y_all[tr]
X_te, y_te = X_all[te], y_all[te]
classes = config['classes']
ens_dir = str(EVAL_DIR / 'ensembles')
samples_dir = EVAL_DIR / 'samples'
samples_dir.mkdir(exist_ok=True)

print(f'EVAL_DIR: {EVAL_DIR}')
print(f'split={SPLIT}, train={len(tr)}, test={len(te)}, classes={classes}')

EVAL_DIR: experiments/fair_eval/20260417_215937
split=0, train=50, test=50, classes=[0, 1]


## 2. VP-SGLD 하이퍼파라미터

In [4]:
BETA = [0, 0.01, 0.1, 1.0]
# BETA = [0, 0.01, 0.1, 1.0, 10.0]
ETA = [1] # 0.1
TAU = [0.0005]
SIGMA_START = [0.1]
N_STEPS = [200]
IGNORE_VARIANCE = [False]
AUTO_BETA = [False]
N_SYN = 250
INCLUDE_SINGLE = True

import itertools
VP_CONFIGS = []
for beta, eta, tau, sigma, steps, iv, ab in itertools.product(
    BETA, ETA, TAU, SIGMA_START, N_STEPS, IGNORE_VARIANCE, AUTO_BETA):
    name = f'vp_b{beta:g}_e{eta:g}_t{tau:g}_s{sigma:g}_T{steps}'
    if iv: name += '_ivT'
    if not ab: name += '_abF'
    VP_CONFIGS.append(dict(name=name, beta=beta, eta=eta, tau=tau,
        sigma_start=sigma, n_steps=steps, auto_beta=ab, ignore_variance=iv))

print(f'{len(VP_CONFIGS)} VP configs, N_SYN={N_SYN}:')
for c in VP_CONFIGS:
    print(f'  {c["name"]}')

4 VP configs, N_SYN=250:
  vp_b0_e1_t0.0005_s0.1_T200_abF
  vp_b0.01_e1_t0.0005_s0.1_T200_abF
  vp_b0.1_e1_t0.0005_s0.1_T200_abF
  vp_b1_e1_t0.0005_s0.1_T200_abF


In [5]:
GPUS = [0, 1, 2, 3]
# GPUS = [0]
print(f'GPUs: {GPUS}')

GPUs: [0, 1, 2, 3]


## 3. SGLD 생성 (단일 split)

In [ ]:
from fair_eval_worker import run_one_sgld_task

tasks = []; tid = 0
for ci, cfg in enumerate(VP_CONFIGS):
    for c in classes:
        out = samples_dir / f'split_{SPLIT}' / cfg['name'] / f'c{c}.npy'
        if out.exists():
            print(f'  SKIP {cfg["name"]} c{c}'); tid += 1; continue
        gpu = GPUS[tid % len(GPUS)]
        tasks.append(('vp', SPLIT, c, {**cfg, '_ci': ci},
                      tr, X_all, y_all, N_SYN, config['seed'], gpu, ens_dir))
        tid += 1
if INCLUDE_SINGLE:
    for c in classes:
        out = samples_dir / f'split_{SPLIT}' / 'tabebm_single' / f'c{c}.npy'
        if out.exists():
            print(f'  SKIP tabebm_single c{c}'); continue
        gpu = GPUS[tid % len(GPUS)]
        tasks.append(('single', SPLIT, c, config['seed'],
                      tr, X_all, y_all, N_SYN, config['seed'], gpu, ens_dir))
        tid += 1

print(f'{len(tasks)} tasks on {len(GPUS)} GPUs')
if tasks:
    t0 = time.time(); done = 0
    with ProcessPoolExecutor(max_workers=len(GPUS)) as ex:
        futs = {ex.submit(run_one_sgld_task, t): t for t in tasks}
        for f in as_completed(futs):
            r = f.result()
            out = samples_dir / f'split_{r["split_i"]}' / r['cfg_name']
            out.mkdir(parents=True, exist_ok=True)
            np.save(out / f'c{r["class_c"]}.npy', r['samples'])
            done += 1
            print(f'  [{done}/{len(tasks)}] {r["cfg_name"]} c{r["class_c"]} '
                  f'({r["dt"]:.0f}s, gpu{r["gpu"]})', flush=True)
    print(f'Done -- {time.time()-t0:.0f}s')
else:
    print('모든 샘플 존재 — skip')

## 4. Eval (단일 split)

In [ ]:
CLASSIFIERS = ['knn', 'lr', 'rf', 'xgboost', 'mlp']

aug_names = [c['name'] for c in VP_CONFIGS]
if INCLUDE_SINGLE: aug_names.append('tabebm_single')
# 실제 존재하는 것만
aug_names = [s for s in aug_names
             if (samples_dir / f'split_{SPLIT}' / s / f'c{classes[0]}.npy').exists()]
SETTINGS = ['baseline'] + aug_names

seed_i = config['seed'] + SPLIT
base = {cn: balanced_accuracy_score(y_te,
        CLASSIFIER_BUILDERS[cn](seed_i).fit(X_tr, y_tr).predict(X_te))*100
        for cn in CLASSIFIERS}

aug = {}
for sn in aug_names:
    smp = {c: np.load(samples_dir/f'split_{SPLIT}'/sn/f'c{c}.npy')[:N_SYN] for c in classes}
    X_a = np.vstack([X_tr]+[smp[c] for c in classes])
    y_a = np.concatenate([y_tr]+[np.full(len(smp[c]),c) for c in classes])
    aug[sn] = (X_a, y_a)

rows = []
for cn in CLASSIFIERS:
    row = {'classifier': cn, 'baseline': base[cn]}
    for sn, (Xa, ya) in aug.items():
        row[sn] = balanced_accuracy_score(y_te,
            CLASSIFIER_BUILDERS[cn](seed_i).fit(Xa, ya).predict(X_te))*100
    rows.append(row)

df = pd.DataFrame(rows)
print(f'split={SPLIT}, N_SYN={N_SYN}')
df

In [ ]:
n_set = len(SETTINGS)
fig, ax = plt.subplots(figsize=(max(10, 1.5*len(CLASSIFIERS)*n_set/3), 5.5))
x = np.arange(len(CLASSIFIERS)); w = 0.8/n_set
for i, s in enumerate(SETTINGS):
    vals = [df.loc[df['classifier']==c, s].values[0] for c in CLASSIFIERS]
    short = s.replace('vp_','').replace('_abF','').replace('tabebm_','')
    ax.bar(x+(i-(n_set-1)/2)*w, vals, w, label=short, capsize=2)
ax.set_xticks(x); ax.set_xticklabels(CLASSIFIERS)
ax.set_ylabel('balanced accuracy (%)')
ax.set_title(f'split={SPLIT}, N_SYN={N_SYN}')
ax.legend(ncol=min(n_set,4), fontsize=7, loc='best')
ax.grid(axis='y', alpha=0.3); plt.tight_layout()
plt.show()

## 5. Heatmap + 생성 샘플 overlay

In [ ]:
from sklearn.decomposition import PCA

ens_split = EVAL_DIR / 'ensembles' / f'split_{SPLIT}'
viz = {}
for c in classes:
    cd = ens_split / f'c{c}'
    cache = None
    for f in cd.iterdir():
        if f.name.startswith('heatmap_pad') and f.suffix == '.npz':
            cache = np.load(f); break
    if cache is None:
        from fair_eval_worker import compute_member_energy
        meta = json.loads((cd / 'meta.json').read_text())
        K = meta['n_ebms']; X_cl = np.load(cd/'class_data.npz')['X_class']
        pca = PCA(2).fit(X_cl); Z = pca.transform(X_cl)
        pad = 5.0; h = 0.2
        zz1 = np.arange(Z.min(0)[0]-pad, Z.max(0)[0]+pad, h)
        zz2 = np.arange(Z.min(0)[1]-pad, Z.max(0)[1]+pad, h)
        ZZ1, ZZ2 = np.meshgrid(zz1, zz2)
        grid = np.column_stack([ZZ1.ravel(), ZZ2.ravel()])
        Xg = pca.inverse_transform(grid).astype(np.float64)
        tasks_hm = [(str(cd), k, Xg, GPUS[k%len(GPUS)]) for k in range(K)]
        E = [None]*K
        with ProcessPoolExecutor(max_workers=len(GPUS)) as ex:
            for f in as_completed({ex.submit(compute_member_energy,t):t[1] for t in tasks_hm}):
                k, e = f.result(); E[k] = e.reshape(ZZ1.shape)
        cache = {'ZZ1': ZZ1, 'ZZ2': ZZ2, 'K': K}
        for k in range(K): cache[f'E_{k}'] = E[k]
        np.savez(cd/f'heatmap_pad{pad}_h{h}.npz', **cache, Z_class=Z,
                 var_ratio=np.array(pca.explained_variance_ratio_.sum()))
        print(f'  class {c}: computed')
    else:
        K = int(cache['K']); ZZ1 = cache['ZZ1']; ZZ2 = cache['ZZ2']
        E = [cache[f'E_{k}'] for k in range(K)]
        X_cl = np.load(cd/'class_data.npz')['X_class']; pca = PCA(2).fit(X_cl)
        print(f'  class {c}: cached')
    E_std = np.stack(E).std(axis=0, ddof=1)
    negs = [np.load(cd/f'ebm_{k}/negatives.npz')['X_neg'] for k in range(K)
            if (cd/f'ebm_{k}/negatives.npz').exists()]
    Xn = np.vstack(negs) if negs else np.empty((0, X_cl.shape[1]))
    viz[c] = {'ZZ1': ZZ1, 'ZZ2': ZZ2, 'std': E_std,
              'Z_real': pca.transform(X_cl),
              'Z_neg': pca.transform(Xn) if len(Xn) else np.empty((0,2)),
              'pca': pca}

In [ ]:
for cfg_name in aug_names:
    fig, axes = plt.subplots(1, len(classes), figsize=(6*len(classes), 5.5))
    if len(classes) == 1: axes = [axes]
    for col, c in enumerate(classes):
        ax = axes[col]; v = viz[c]
        ax.contourf(v['ZZ1'], v['ZZ2'], v['std'], levels=20, cmap='viridis')
        ax.scatter(*v['Z_real'].T, c='black', s=18, alpha=0.8, zorder=3, label='pos')
        ax.scatter(*v['Z_neg'].T, c='white', s=40, marker='x', linewidths=1.5, zorder=3, label='neg')
        path = samples_dir / f'split_{SPLIT}' / cfg_name / f'c{c}.npy'
        if path.exists():
            Zg = v['pca'].transform(np.load(path)[:N_SYN])
            ax.scatter(Zg[:,0], Zg[:,1], c='magenta', s=12, alpha=0.6,
                       zorder=4, edgecolors='none', label=f'gen ({len(Zg)})')
        ax.set_title(f'class {c}', fontsize=11)
        ax.set_aspect('equal')
        if col==0: ax.legend(fontsize=8)
    short = cfg_name.replace('vp_','').replace('_abF',' (ab=F)').replace('tabebm_','te_')
    fig.suptitle(f'{short}  (split {SPLIT})', fontsize=12)
    plt.tight_layout(); plt.show()